# Controlled comparison of labeling methods on noisy samples

This notebook mirrors the data-preparation and ground-truth construction of
`compare_soft_labeling_on_pure_samples.ipynb`, but evaluates the labeling
methods on **noisy samples** (samples that contain a mixture of two cell types)
instead of pure samples.

In [ ]:
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Union

import numpy as np
import pandas as pd
import json
import pysam
from tqdm import tqdm
import matplotlib.pyplot as plt

from utils import (
    filename2ctype,
    nested_defaultdict_to_dict,
    get_idx_from_sig,
    get_sig_from_idx,
    nrmse,
    compute_counts_sig_per_ctype_from_dataset,
    extract_complete_dataset,
    compute_num_reads_in_dataset,
    generate_bootstrap_sample_from_dataset,
    compute_pgt_sig_prior_from_counts,
    compute_nrmse_per_sig_length,
)
from methods import (
    naive_sl_without_pooling,
    generalized_naive_sl_without_pooling,
    sl_with_simple_archetypes,
)

## Data preparation

### Setting selection

We study two cell types and a single DMR under one of two settings:

- **`differentiated`**: one of the two selected cell types is the *target* cell
  type of the DMR (i.e. the DMR is differentially methylated for it), while the
  other is not. The two cell types are therefore distinguishable at this DMR.
- **`non_differentiated`**: *neither* of the two selected cell types is the
  target cell type of the DMR. The two cell types are not expected to be
  distinguishable at this DMR.

Change the single `SETTING` flag below to switch between the two settings for
the whole notebook.

In [ ]:
# Single flag controlling the whole notebook.
SETTING = "differentiated"  # one of {"differentiated", "non_differentiated"}
assert SETTING in {"differentiated", "non_differentiated"}

N_CTYPES = 2  # we always select exactly two cell types

### Selection of the ctypes and DMRs

First we load the file containing the coverage per ctype per dmr.

In [ ]:
df_coverage_per_ctype_per_dmr = pd.read_csv(Path("./data/region_coverage.csv"))
df_coverage_per_ctype_per_dmr.columns = [
    col if col != "Ovary+Endom-Ep" else "Ovary-Ep"
    for col in df_coverage_per_ctype_per_dmr.columns
]
atlas = pd.read_csv(Path("../../Data/Atlas.U25.l4.hg38.full.tsv"), sep="\t")
# add target column to df_coverage_per_ctype_per_dmr
map_region_key_to_target = atlas.set_index(["chr", "start", "end"])["target"].to_dict()
df_coverage_per_ctype_per_dmr["target"] = df_coverage_per_ctype_per_dmr.apply(
    lambda row: map_region_key_to_target.get((row["chr"], row["start"], row["end"])),
    axis=1,
)

labels_dict = json.load(open(Path("../../App/labels_dict.json"), "r", encoding="utf-8"))
labels_dict = {int(k): v for k, v in labels_dict.items()}
labels_list = [labels_dict[i] for i in range(len(labels_dict))]

We now select the two cell types and the DMR $d$ that maximize the minimum
coverage of the two selected cell types, under the constraints imposed by the
chosen `SETTING`:

- In the `differentiated` setting, cell type 0 is forced to be the target cell
  type of the DMR and cell type 1 is the non-target cell type with the highest
  coverage at that DMR.
- In the `non_differentiated` setting, both cell types are chosen among the
  non-target cell types with the highest coverage at that DMR.

In [ ]:
best_dmr_idx = None
best_min_coverage = -1.0
best_ctypes = None

for dmr_idx, row in df_coverage_per_ctype_per_dmr.iterrows():
    if row["target"] not in labels_list:
        continue
    target_ctype_idx = labels_list.index(row["target"])
    coverage_per_ctype = row[labels_list].values.astype(float)

    if SETTING == "differentiated":
        # ctype 0 = target ctype, ctype 1 = best-covered non-target ctype
        coverage_excl_target = coverage_per_ctype.copy()
        coverage_excl_target[target_ctype_idx] = -1.0
        other_ctype_idx = int(np.argmax(coverage_excl_target))
        selected_idx = [target_ctype_idx, other_ctype_idx]
    else:  # non_differentiated
        # both ctypes = the two best-covered non-target ctypes
        coverage_excl_target = coverage_per_ctype.copy()
        coverage_excl_target[target_ctype_idx] = -1.0
        selected_idx = list(np.argsort(coverage_excl_target)[-N_CTYPES:])

    min_coverage = coverage_per_ctype[selected_idx].min()
    if min_coverage > best_min_coverage:
        best_min_coverage = min_coverage
        best_dmr_idx = dmr_idx
        best_ctypes = [labels_list[i] for i in selected_idx]

print(f"Setting: {SETTING}")
print(
    f"Best dmr: {best_dmr_idx}, Best ctypes: {best_ctypes}, "
    f"Best min coverage: {best_min_coverage}"
)
print(
    f"Target ctype of the dmr: {df_coverage_per_ctype_per_dmr.loc[best_dmr_idx, 'target']}"
)
print(
    f"Expected number of reads in the raw dataset: "
    f"{df_coverage_per_ctype_per_dmr.loc[best_dmr_idx, best_ctypes].sum()}"
)

In [ ]:
selected_ctypes = best_ctypes
selected_dmr_idx = best_dmr_idx
DMR_START_CPG = df_coverage_per_ctype_per_dmr.loc[selected_dmr_idx, "startCpG"]
DMR_FETCH_START = DMR_START_CPG - 50
DMR_END_CPG = df_coverage_per_ctype_per_dmr.loc[selected_dmr_idx, "endCpG"]
DMR_CHR = df_coverage_per_ctype_per_dmr.loc[selected_dmr_idx, "chr"]
print(
    df_coverage_per_ctype_per_dmr.loc[
        selected_dmr_idx,
        ["chr", "start", "end", "startCpG", "endCpG", "target"] + selected_ctypes,
    ]
)

### Extraction of reads

Reads are organized in a nested dictionary. The first level of the dictionary is
the cell type, the second level is the number of CpG sites in the reads, and the
third level are the individual starting positions. The values are dictionaries
mapping the pattern of methylation to the number of reads with that pattern.

We first extract the raw reads from the files (so the pattern may contain unknown
values). We then split the reads with unknown values into smaller reads
(separated by the unknown values). Finally, we create an additional "complete"
dataset where we add to the reads of size n all the subsets of reads with
size > n (to be used position per position only).

In [ ]:
# get the list of files paths per ctype
PATH_TO_DATA = Path(
    "/staging/leuven/stg_00118/methylDL/data/loyfer2023/hg38/data/GSE186458"
)
files_per_ctype = defaultdict(list)
for file in PATH_TO_DATA.glob("*.pat.gz"):
    ctype = filename2ctype(file.name)
    if ctype in selected_ctypes:
        files_per_ctype[ctype].append(file)
files_per_ctype = dict(files_per_ctype)

In [ ]:
# Extract the raw reads dataset for the selected dmr and selected ctypes
raw_reads_dataset = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
)
n_loops = sum(len(files) for files in files_per_ctype.values())
with tqdm(total=n_loops, desc="Extracting reads") as pbar:
    for ctype, files in files_per_ctype.items():
        for file in files:
            csi_file_path = str(file) + ".csi"
            tbx = pysam.TabixFile(str(file), index=csi_file_path)
            for row in tbx.fetch(DMR_CHR, DMR_FETCH_START, DMR_END_CPG + 1):
                parts = row.split("\t")
                read_start = int(parts[1])
                pattern = parts[2]
                n_reads = int(parts[3])
                read_end = read_start + len(pattern)  # exclusive
                # crop the read to the dmr region
                if read_end < DMR_START_CPG:
                    continue  # the read ends before the dmr start
                if read_start >= DMR_END_CPG:
                    continue  # the read starts after the dmr end
                cropped_pattern_start = max(read_start, DMR_START_CPG)
                cropped_pattern_end = min(read_end, DMR_END_CPG)
                cropped_pattern = pattern[
                    cropped_pattern_start
                    - read_start : cropped_pattern_end
                    - read_start
                ]
                if len(cropped_pattern) == 0:
                    continue  # the read does not overlap with the dmr
                # update the count of the cropped pattern in the raw reads dataset
                cropped_pattern = cropped_pattern.replace("C", "1").replace("T", "0")
                raw_reads_dataset[ctype][len(cropped_pattern)][cropped_pattern_start][
                    cropped_pattern
                ] += n_reads
                pbar.update(1)
n_reads_in_raw_dataset = compute_num_reads_in_dataset(raw_reads_dataset)
print(f"Number of reads in raw dataset: {n_reads_in_raw_dataset}")

In [ ]:
# Create the original read dataset by splitting the patterns at the unknown positions "."
og_dataset = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
)
for ctype, length_dict in raw_reads_dataset.items():
    for read_length in sorted(length_dict.keys()):  # start from the shortest reads
        for start, pattern_counts in length_dict[read_length].items():
            for pattern, n_reads in pattern_counts.items():
                # split the pattern at the unknown positions "."
                subpatterns = pattern.split(".")
                subpattern_starts = []
                current_pos = start
                for subpattern in subpatterns:
                    subpattern_starts.append(current_pos)
                    current_pos += len(subpattern)
                # update the count of the subpatterns in the reads dataset
                for subpattern, subpattern_start in zip(subpatterns, subpattern_starts):
                    if len(subpattern) == 0:
                        continue  # skip empty subpatterns
                    og_dataset[ctype][len(subpattern)][subpattern_start][
                        subpattern
                    ] += n_reads
n_reads_in_og_dataset = compute_num_reads_in_dataset(og_dataset)
print(
    f"Number of reads in original dataset (reads splitted at unknown positions): "
    f"{n_reads_in_og_dataset}"
)

In [ ]:
# Create a complete dataset by splitting long patterns into all possible subpatterns
complete_og_dataset = extract_complete_dataset(og_dataset)
print(
    f"Number of reads in complete original dataset: "
    f"{compute_num_reads_in_dataset(complete_og_dataset)}"
)

Now we create the dict of all pairs (pattern start, pattern length) (over all
ctypes), which represent the possible positions of signatures.

In [ ]:
# first we identify the start_positions common to all ctypes
start_positions_per_ctype = defaultdict(set)
for ctype, length_dict in complete_og_dataset.items():
    for read_length, start_dict in length_dict.items():
        for start in start_dict.keys():
            start_positions_per_ctype[ctype].add(start)
common_start_positions = set.intersection(*start_positions_per_ctype.values())

# then we identify the (start, read_length) pairs common to all ctypes
signature_positions_per_ctype = defaultdict(lambda: defaultdict(set))
for ctype, length_dict in complete_og_dataset.items():
    for read_length, start_dict in length_dict.items():
        for start in start_dict.keys():
            if start not in common_start_positions:
                continue
            signature_positions_per_ctype[ctype][start].add(read_length)
signature_positions = defaultdict(set)
for start in common_start_positions:
    signature_positions[start] = set.intersection(
        *[signature_positions_per_ctype[ctype][start] for ctype in selected_ctypes]
    )
signature_positions = nested_defaultdict_to_dict(signature_positions)
del common_start_positions
del start_positions_per_ctype
del signature_positions_per_ctype
print("Signature positions (pattern start, pattern length) pairs common to all ctypes:")
signature_positions

## Ground truth $P(\mathrm{sig}\mid\mathrm{ctype})$ for the whole dataset

As in `compare_soft_labeling_on_pure_samples.ipynb`, we construct the pseudo
ground truth (PGT) for $P(\mathrm{signature}\mid\mathrm{ctype})$ over the whole
dataset by counting on the complete dataset (there is no pooling of counts
between signatures) and normalizing per cell type with the naive soft labels
without pooling.

In [ ]:
# compute counts of signatures per ctype
complete_og_counts_sig_per_ctype = compute_counts_sig_per_ctype_from_dataset(
    complete_og_dataset, signature_positions, selected_ctypes, N_CTYPES
)  # with the complete dataset
og_counts_sig_per_ctype = compute_counts_sig_per_ctype_from_dataset(
    og_dataset, signature_positions, selected_ctypes, N_CTYPES
)  # with the reads dataset

# compute the PGT P(s)
complete_og_pgt_sig_prior = compute_pgt_sig_prior_from_counts(
    complete_og_counts_sig_per_ctype, signature_positions
)  # with the complete dataset
og_pgt_sig_prior = compute_pgt_sig_prior_from_counts(
    og_counts_sig_per_ctype, signature_positions
)  # with the reads dataset

# compute the ground truth P(sig | ctype) with naive soft labels without pooling
# (warning is expected here as signatures with zero counts in all ctypes lead to
# division by zero)
pgt_naive_sl_sig_given_ctype = naive_sl_without_pooling(
    complete_og_counts_sig_per_ctype
)

## Construction of noisy samples

A **noisy sample** is a mixture of the two selected cell types: a fraction
$p_{b,c}$ of its reads is drawn from the pure read distribution of cell type $c$.

We provide two helpers:

- `build_ctype_prop_per_sample(noise_level, n_samples)` builds the per-sample
  mixing proportions. Samples alternate which cell type is dominant; the
  `noise_level` is the contamination fraction of the non-dominant cell type
  (`0.0` = pure samples, `0.5` = fully mixed 50/50 samples). A larger
  `noise_level` therefore corresponds to noisier samples.
- `generate_noisy_samples_counts(...)` draws reads for each sample from the pure
  per-ctype datasets according to those proportions and returns the per-sample
  signature counts in the `{start: {read_length: array(n_samples, 2**read_length)}}`
  format expected by the labeling methods.

In [ ]:
def build_ctype_prop_per_sample(
    noise_level: float, n_samples: int, n_ctypes: int = N_CTYPES
) -> np.ndarray:
    """Build per-sample mixing proportions for a given noise level.

    Samples alternate the dominant cell type. The dominant cell type gets
    proportion ``1 - noise_level`` and the remaining ``noise_level`` is spread
    equally over the other cell types.
    """
    props = np.zeros((n_samples, n_ctypes))
    for b in range(n_samples):
        dominant = b % n_ctypes
        props[b, dominant] = 1.0 - noise_level
        others = [c for c in range(n_ctypes) if c != dominant]
        for c in others:
            props[b, c] = noise_level / len(others)
    return props


def generate_noisy_samples_counts(
    source_dataset,
    selected_ctypes_,
    ctype_prop_per_sample: np.ndarray,
    sample_sizes: np.ndarray,
    sig_positions,
    n_ctypes: int,
):
    """Generate per-sample signature counts for noisy (mixed) samples.

    For each sample b and each cell type c, ``round(p_{b,c} * sample_size_b)``
    reads are drawn (with replacement) from the pure read distribution of c, and
    the resulting signature counts are summed across cell types (the cell type of
    origin is unknown at the sample level).

    Returns a dict ``{start: {read_length: np.ndarray of shape (n_samples, 2**read_length)}}``.
    """
    n_samples = ctype_prop_per_sample.shape[0]
    counts_sig_per_sample = {
        start: {
            read_length: np.zeros((n_samples, 2**read_length), dtype=int)
            for read_length in read_lengths
        }
        for start, read_lengths in sig_positions.items()
    }
    for b in range(n_samples):
        for c_idx, ctype in enumerate(selected_ctypes_):
            n_reads_c = int(round(ctype_prop_per_sample[b, c_idx] * sample_sizes[b]))
            if n_reads_c <= 0:
                continue
            single_ctype_dataset = {ctype: source_dataset[ctype]}
            sampled = generate_bootstrap_sample_from_dataset(
                single_ctype_dataset, dataset_size_to_sample=n_reads_c
            )
            sampled_counts = compute_counts_sig_per_ctype_from_dataset(
                sampled, sig_positions, [ctype], 1
            )
            for start, read_lengths in sampled_counts.items():
                for read_length, counts in read_lengths.items():
                    counts_sig_per_sample[start][read_length][b] += counts[0]
    return counts_sig_per_sample

## Experiment 1: deconvolution with oracle proportions

We construct noisy samples and deconvolve them with two methods, providing each
method with the **oracle** mixing proportions (the methods do not re-estimate
them):

- `generalized_naive_sl_without_pooling` (to be implemented later),
- `sl_with_simple_archetypes`.

We then plot the distribution of the NRMSE between the estimated
$P(\mathrm{sig}\mid\mathrm{ctype})$ and the ground-truth
$P(\mathrm{sig}\mid\mathrm{ctype})$ for each method, exactly as in
`compare_soft_labeling_on_pure_samples.ipynb`, except that the columns now vary a
measure of the **noise in the samples** (the contamination fraction) instead of
the dataset size.

In [ ]:
N_SIMULATIONS = 50
N_NOISY_SAMPLES = 20  # number of noisy samples jointly deconvolved per simulation
SAMPLE_SIZE = 2000  # number of reads per noisy sample
noise_grid = np.array([0.0, 0.1, 0.25, 0.4])
min_count_threshold_per_ctype_for_nrmse = 1

methods_exp1 = ["generalized_naive_sl_without_pooling", "sl_with_simple_archetypes"]

# Reference counts used to decide which (sig, ctype) pairs are considered in the
# NRMSE (those with ground-truth support). Kept fixed across simulations.
ref_counts_for_nrmse = og_counts_sig_per_ctype

results_exp1_per_noise_per_read_length_per_method = {
    noise_level: {
        read_length: {method: [] for method in methods_exp1}
        for read_length in range(1, 6)
    }
    for noise_level in noise_grid
}

n_loops = len(noise_grid) * N_SIMULATIONS
with tqdm(total=n_loops, desc="Exp 1: oracle proportions") as pbar:
    for noise_level in noise_grid:
        ctype_prop_per_sample = build_ctype_prop_per_sample(
            noise_level, N_NOISY_SAMPLES, N_CTYPES
        )
        for sim_id in range(N_SIMULATIONS):
            sample_sizes = np.full(N_NOISY_SAMPLES, SAMPLE_SIZE, dtype=int)
            counts_sig_per_sample = generate_noisy_samples_counts(
                og_dataset,
                selected_ctypes,
                ctype_prop_per_sample,
                sample_sizes,
                signature_positions,
                N_CTYPES,
            )

            # both methods receive the oracle proportions and do not re-estimate them
            generalized_sig_given_ctype = generalized_naive_sl_without_pooling(
                counts_sig_per_sample,
                ctype_prop_per_sample,
                reestimate_ctype_prop_per_sample=False,
            )
            simple_archetypes_sig_given_ctype = sl_with_simple_archetypes(
                counts_sig_per_sample,
                ctype_prop_per_sample,
                reestimate_ctype_proba_per_sample=False,
                max_num_iterations=100,
                delta_tol=1e-4,
            )

            for method_name, estimated in [
                ("generalized_naive_sl_without_pooling", generalized_sig_given_ctype),
                ("sl_with_simple_archetypes", simple_archetypes_sig_given_ctype),
            ]:
                nrmse_per_length, _ = compute_nrmse_per_sig_length(
                    estimated,
                    pgt_naive_sl_sig_given_ctype,
                    counts_sig_per_ctype=ref_counts_for_nrmse,
                    min_count_threshold_per_ctype=min_count_threshold_per_ctype_for_nrmse,
                )
                for read_length in range(1, 6):
                    if read_length in nrmse_per_length:
                        results_exp1_per_noise_per_read_length_per_method[noise_level][
                            read_length
                        ][method_name].append(nrmse_per_length[read_length])
            pbar.update(1)

In [ ]:
from matplotlib.patches import Patch

method_labels_exp1 = ["Generalized naive\n(no pooling)", "Simple\narchetypes"]
colors_exp1 = ["tab:blue", "tab:red"]

fig, axes = plt.subplots(
    5, len(noise_grid), figsize=(4 * len(noise_grid), 4 * 5), sharey="row"
)

for row_idx, read_length in enumerate(range(1, 6)):
    for col_idx, noise_level in enumerate(noise_grid):
        ax = axes[row_idx, col_idx]
        data_to_plot = [
            results_exp1_per_noise_per_read_length_per_method[noise_level][read_length][
                method
            ]
            for method in methods_exp1
        ]
        bp = ax.boxplot(data_to_plot, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors_exp1):
            patch.set_facecolor(color)
        for median in bp["medians"]:
            median.set_color("black")
        ax.set_ylim(0, None)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.set_xticks([])
        if row_idx == 0:
            ax.set_title(f"Noise level (contamination)\n= {noise_level:.2f}")
        ax.set_ylabel(f"Read length={read_length}\nNRMSE" if col_idx == 0 else "")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp1, method_labels_exp1)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.85, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Exp 1 (oracle proportions): NRMSE of estimated P(signature | ctype) vs sample "
    "noise level, per pattern length and method"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"#noisy samples: {N_NOISY_SAMPLES}, sample size: {SAMPLE_SIZE}, "
    f"#simulations: {N_SIMULATIONS}",
    y=1.0,
)
plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.show()

## Experiment 2: deconvolution with a noisy estimate of the proportions

Same setup as Experiment 1, but the methods now receive a **noisy estimate** of
the mixing proportions instead of the oracle values, and are allowed to
re-estimate them (`reestimate_ctype_proba_per_sample=True`). We add Gaussian
noise to the true proportions and renormalize to obtain the initial estimate
passed to each method.

We plot two quantities:

- the NRMSE between the estimated $P(\mathrm{sig}\mid\mathrm{ctype})$ and the
  ground truth (as in Experiment 1), and
- the NRMSE between the proportions estimated by the methods and the true mixing
  proportions of the samples.

> **Note**: for the proportion NRMSE to be meaningful, the methods are expected
> to return, when `reestimate_...=True`, a tuple
> `(P(sig | ctype), estimated_ctype_proportions)`. The helper `run_method` below
> handles both return conventions: if a method returns only
> `P(sig | ctype)`, the proportion estimate falls back to the (noisy) input.

In [ ]:
PROP_ESTIMATE_NOISE_STD = 0.1  # std of the Gaussian noise added to the true proportions


def perturb_proportions(true_props: np.ndarray, noise_std: float, rng) -> np.ndarray:
    """Return a noisy estimate of the proportions (clipped and renormalized)."""
    noisy = true_props + rng.normal(0.0, noise_std, size=true_props.shape)
    noisy = np.clip(noisy, 1e-3, None)
    noisy = noisy / noisy.sum(axis=1, keepdims=True)
    return noisy


def run_method(method_name: str, counts_sig_per_sample, init_props: np.ndarray):
    """Run a labeling method with re-estimation of the proportions.

    Returns a tuple ``(proba_sig_given_ctype, estimated_props)``. If the method
    only returns ``proba_sig_given_ctype``, the estimated proportions fall back to
    ``init_props``.
    """
    if method_name == "generalized_naive_sl_without_pooling":
        out = generalized_naive_sl_without_pooling(
            counts_sig_per_sample,
            init_props,
            reestimate_ctype_prop_per_sample=True,
        )
    else:
        out = sl_with_simple_archetypes(
            counts_sig_per_sample,
            init_props,
            reestimate_ctype_proba_per_sample=True,
            max_num_iterations=100,
            delta_tol=1e-4,
        )
    if isinstance(out, tuple):
        proba_sig_given_ctype, estimated_props = out
    else:
        proba_sig_given_ctype, estimated_props = out, init_props
    return proba_sig_given_ctype, estimated_props

In [ ]:
methods_exp2 = ["generalized_naive_sl_without_pooling", "sl_with_simple_archetypes"]

rng = np.random.default_rng(0)

results_exp2_sig_given_ctype = {
    noise_level: {
        read_length: {method: [] for method in methods_exp2}
        for read_length in range(1, 6)
    }
    for noise_level in noise_grid
}
results_exp2_proportions = {
    noise_level: {method: [] for method in methods_exp2} for noise_level in noise_grid
}

n_loops = len(noise_grid) * N_SIMULATIONS
with tqdm(total=n_loops, desc="Exp 2: noisy proportion estimate") as pbar:
    for noise_level in noise_grid:
        ctype_prop_per_sample = build_ctype_prop_per_sample(
            noise_level, N_NOISY_SAMPLES, N_CTYPES
        )
        for sim_id in range(N_SIMULATIONS):
            sample_sizes = np.full(N_NOISY_SAMPLES, SAMPLE_SIZE, dtype=int)
            counts_sig_per_sample = generate_noisy_samples_counts(
                og_dataset,
                selected_ctypes,
                ctype_prop_per_sample,
                sample_sizes,
                signature_positions,
                N_CTYPES,
            )

            # noisy estimate of the proportions given to the methods
            noisy_prop_estimate = perturb_proportions(
                ctype_prop_per_sample, PROP_ESTIMATE_NOISE_STD, rng
            )

            for method_name in methods_exp2:
                estimated_sig_given_ctype, estimated_props = run_method(
                    method_name, counts_sig_per_sample, noisy_prop_estimate
                )

                # NRMSE of P(sig | ctype)
                nrmse_per_length, _ = compute_nrmse_per_sig_length(
                    estimated_sig_given_ctype,
                    pgt_naive_sl_sig_given_ctype,
                    counts_sig_per_ctype=ref_counts_for_nrmse,
                    min_count_threshold_per_ctype=min_count_threshold_per_ctype_for_nrmse,
                )
                for read_length in range(1, 6):
                    if read_length in nrmse_per_length:
                        results_exp2_sig_given_ctype[noise_level][read_length][
                            method_name
                        ].append(nrmse_per_length[read_length])

                # NRMSE of the estimated proportions vs the true proportions
                proportion_nrmse = nrmse(ctype_prop_per_sample, estimated_props)
                results_exp2_proportions[noise_level][method_name].append(
                    proportion_nrmse
                )
            pbar.update(1)

In [ ]:
# Plot the NRMSE of P(sig | ctype) for Experiment 2
method_labels_exp2 = ["Generalized naive\n(no pooling)", "Simple\narchetypes"]
colors_exp2 = ["tab:blue", "tab:red"]

fig, axes = plt.subplots(
    5, len(noise_grid), figsize=(4 * len(noise_grid), 4 * 5), sharey="row"
)

for row_idx, read_length in enumerate(range(1, 6)):
    for col_idx, noise_level in enumerate(noise_grid):
        ax = axes[row_idx, col_idx]
        data_to_plot = [
            results_exp2_sig_given_ctype[noise_level][read_length][method]
            for method in methods_exp2
        ]
        bp = ax.boxplot(data_to_plot, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors_exp2):
            patch.set_facecolor(color)
        for median in bp["medians"]:
            median.set_color("black")
        ax.set_ylim(0, None)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.set_xticks([])
        if row_idx == 0:
            ax.set_title(f"Noise level (contamination)\n= {noise_level:.2f}")
        ax.set_ylabel(f"Read length={read_length}\nNRMSE" if col_idx == 0 else "")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp2, method_labels_exp2)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.85, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Exp 2 (noisy proportion estimate): NRMSE of estimated P(signature | ctype) vs "
    "sample noise level, per pattern length and method"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"prop. estimate noise std: {PROP_ESTIMATE_NOISE_STD}, "
    f"#simulations: {N_SIMULATIONS}",
    y=1.0,
)
plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.show()

In [ ]:
# Plot the NRMSE of the estimated proportions for Experiment 2
fig, axes = plt.subplots(
    1, len(noise_grid), figsize=(4 * len(noise_grid), 4), sharey=True
)
if len(noise_grid) == 1:
    axes = [axes]

for col_idx, noise_level in enumerate(noise_grid):
    ax = axes[col_idx]
    data_to_plot = [
        results_exp2_proportions[noise_level][method] for method in methods_exp2
    ]
    bp = ax.boxplot(data_to_plot, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors_exp2):
        patch.set_facecolor(color)
    for median in bp["medians"]:
        median.set_color("black")
    ax.set_ylim(0, None)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_xticks([])
    ax.set_title(f"Noise level (contamination)\n= {noise_level:.2f}")
    if col_idx == 0:
        ax.set_ylabel("NRMSE of estimated proportions")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp2, method_labels_exp2)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.9, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Exp 2 (noisy proportion estimate): NRMSE of the estimated ctype proportions vs "
    "sample noise level, per method"
    f"\nSetting: {SETTING}, prop. estimate noise std: {PROP_ESTIMATE_NOISE_STD}, "
    f"#simulations: {N_SIMULATIONS}",
    y=1.05,
)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()